# Setup NemoHermes and VSS on the Brev instance

This notebook is meant to run **on the Brev instance itself**. It walks through the full Brev-side flow for **NemoClaw / OpenShell + Hermes** and the **Video Search and Summarization (VSS)** blueprint: configure or reuse a Hermes sandbox, verify the policy and imported skills, pin the tested Docker package versions, prepare the host prerequisites for local NIM-backed profiles, start the host-side VSS Orchestrator MCP server, then deploy and manage VSS from NemoHermes by chatting with the agent.

**Default launchable:**  
[video-search-and-summarization-blueprint](https://brev.nvidia.com/launchable/deploy/now?launchableID=env-2tYIjRXL4eMCbH9Az8mJC5WPAI4)

If you are using that launchable, the VM is expected to already have the main prerequisites in place. If not, use the preflight cells below to confirm what is missing.

**Required prerequisites**

- Run this notebook with **Python 3.11 or newer**. The MCP helper uses Python 3.11 standard-library APIs.
- Make sure the intended VSS checkout is the one resolved by `VSS_REPO_DIR`. By default this notebook uses `~/video-search-and-summarization`; set the `VSS_REPO_DIR` environment variable before launching Jupyter if your checkout lives elsewhere or the host has multiple clones.
- If you are re-onboarding a host that previously ran an older OpenShell gateway, follow the NemoClaw sandbox lifecycle upgrade guidance: <https://docs.nvidia.com/nemoclaw/manage-sandboxes/lifecycle>.

**What this notebook covers**

1. Set required keys and notebook options.
2. Run preflight checks for the local repo, scripts, policy file, MCP helper, and host prerequisites, then pin Docker to the tested package versions.
3. Create or reuse the NemoHermes sandbox, configure the chosen Hermes model provider, apply the VSS policy, install VSS skills, upload Hermes workspace instructions, and verify the Hermes API and optionally enable the Hermes dashboard.
4. *(Optional)* Verify the live sandbox state, active policy metadata, Hermes CLI availability, workspace instructions, and installed skills.
5. Prepare the host for local NIM-backed VSS profiles by installing/configuring NGC CLI, logging Docker into `nvcr.io`, and preparing the `services/agent/` environment.
6. Start the host-side VSS Orchestrator MCP server, connect to NemoHermes and smoke-test it, then deploy and manage VSS by asking the agent to use the VSS Orchestrator MCP tools. Teardown of a deployed VSS stack is also done from the agent.
7. *(Optional)* Verify sandbox-to-host reachability through `host.openshell.internal`.

**Security:** prefer `NVIDIA_API_KEY` and `NGC_CLI_API_KEY` from environment variables or Brev secrets. Do **not** commit notebook outputs that contain credentials or live access tokens.

For standalone Hermes on the host, skip this notebook and add this repository's `skills/` directory to Hermes `skills.external_dirs`. OpenClaw migration is optional and only needed to bring existing OpenClaw user data into Hermes.


## 1. Settings

Configure the notebook in two parts:

1. **Required settings** — credentials and hardware profile that every deployment needs.
2. **Choose ONE Hermes model provider** — pick exactly one of (a) a SOTA cloud model, (b) a locally-hosted OpenAI-compatible model, or (c) a model from build.nvidia.com.

> Run section 1.1, then run **exactly one** of the (a) / (b) / (c) cells. Section 1.3 holds advanced defaults you can usually leave alone.

<span style="color:red"><strong>Important:</strong> set <code>NGC_CLI_API_KEY</code> below, and at least one of <code>NVIDIA_API_KEY</code> / <code>COMPATIBLE_API_KEY</code> via the provider cell you pick. Credentials can also come from the environment or Brev secrets.</span>


### 1.1 Required settings

These apply to every deployment:

- `NGC_CLI_API_KEY` — NVIDIA legacy API key used to pull VSS container artifacts from `nvcr.io`.
- `HARDWARE_PROFILE` — selects the hardware-specific overlay.


In [ ]:
# ================== Required settings (always set) ==================
NGC_CLI_API_KEY = ""               # NVIDIA Legacy API key - used to pull VSS artifacts from nvcr.io
HARDWARE_PROFILE = "RTXPRO6000BW"  # DGX-SPARK | RTXPRO6000BW | H100 | L40S | OTHER

# ================== Shared Hermes model vars (overridden by ONE of (a)/(b)/(c) in 1.2) ==================
NVIDIA_API_KEY = ""
NEMOCLAW_ENDPOINT_URL = ""
NEMOCLAW_MODEL = ""
COMPATIBLE_API_KEY = ""


### 1.2 Choose ONE Hermes model provider

Run **exactly one** of the cells below.

| Option | When to use | Sets |
|---|---|---|
| **(a) SOTA cloud model** | Best agent quality through any OpenAI-compatible cloud API. | `NEMOCLAW_ENDPOINT_URL`, `NEMOCLAW_MODEL`, `COMPATIBLE_API_KEY` |
| **(b) Local OpenAI-compatible model** | Self-hosted on this box or LAN | `NEMOCLAW_ENDPOINT_URL`, `NEMOCLAW_MODEL`, `COMPATIBLE_API_KEY` |
| **(c) build.nvidia.com NVIDIA-hosted model** | Default path — uses NVIDIA's hosted Nemotron via `integrate.api.nvidia.com`. | `NVIDIA_API_KEY`, optionally `NEMOCLAW_MODEL` |


#### (a) SOTA cloud model — *recommended for best agent quality*


In [ ]:
# (a) SOTA cloud model - fill in these three values, then run.
NEMOCLAW_ENDPOINT_URL = ""  # OpenAI-compatible base URL, e.g. "https://api.openai.com/v1/"
NEMOCLAW_MODEL        = ""  # Model id at that endpoint
COMPATIBLE_API_KEY    = ""  # Bearer token


#### (b) Local OpenAI-compatible model — *self-hosted / air-gapped*

Point Hermes at a local OpenAI-compatible server you've already started on this host or your LAN.

> `COMPATIBLE_API_KEY` is technically required by the installer (it errors if blank), but most local servers ignore the value — any non-empty placeholder works.
> From inside the NemoClaw sandbox, the host is reachable as `host.openshell.internal`. If your server binds only to `127.0.0.1` on the host, rebind it to `0.0.0.0` — the sandbox cannot reach a loopback-only socket via `host.openshell.internal`.


In [ ]:
# (b) Local OpenAI-compatible model - fill in to match your local server, then run.
NEMOCLAW_ENDPOINT_URL = "http://host.openshell.internal:8000/v1"
NEMOCLAW_MODEL        = "nvidia/nemotron-3-super-120b-a12b"  # the id your local server reports
COMPATIBLE_API_KEY    = "EMPTY"                              # most local servers ignore this; must be non-empty


#### (c) build.nvidia.com NVIDIA-hosted model — *default, zero-setup*

Uses NVIDIA's hosted model catalog via `integrate.api.nvidia.com`. Only `NVIDIA_API_KEY` is required; `NEMOCLAW_MODEL` defaults to `nvidia/nemotron-3-super-120b-a12b` inside the installer if left blank.

> Get a key at <https://build.nvidia.com> (format `nvapi-...`). To use a different hosted model, set `NEMOCLAW_MODEL` to its build.nvidia.com id.


In [ ]:
# (c) build.nvidia.com - set NVIDIA_API_KEY; clear the custom-endpoint vars so the installer picks NEMOCLAW_PROVIDER=build.
NVIDIA_API_KEY = ""  # "nvapi-..." from https://build.nvidia.com
NEMOCLAW_ENDPOINT_URL = ""
COMPATIBLE_API_KEY = ""
NEMOCLAW_MODEL = "nvidia/nemotron-3-super-120b-a12b"  # leave blank to use the init script default


### 1.3 Advanced settings (defaults — usually leave alone)

Pinned installer ref, NemoHermes API/dashboard settings, and VSS LLM/VLM overrides. `EXTERNAL_IP` falls back to `hostname -I` when blank.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path


# ================== Default NemoHermes settings ==================
NEMOCLAW_INSTALL_REF = "v0.0.48"  # NemoClaw installer pin
NEMOHERMES_API_PORT = 8642
NEMOCLAW_HERMES_DASHBOARD = False  # Set True to enable the optional Hermes web dashboard
NEMOCLAW_HERMES_DASHBOARD_PORT = 9119

# ================== VSS settings ==================
VSS_LLM_NAME = ""
VSS_LLM_ENDPOINT_URL = ""
VSS_LLM_MODEL_TYPE = ""
VSS_LLM_ENABLE_THINKING = ""
VSS_OPENAI_API_KEY = ""            # blank => fall back to NVIDIA_API_KEY (typical for integrate.api.nvidia.com)

# Remote VLM - set VSS_VLM_ENDPOINT_URL non-empty to force VLM_MODE=remote in generated.env
VSS_VLM_NAME = ""
VSS_VLM_ENDPOINT_URL = ""
VSS_VLM_MODEL_TYPE = ""

LLM_DEVICE_ID = "0"
VLM_DEVICE_ID = "1"
EXTERNAL_IP = ""                    # blank => resolve from `hostname -I`


# ================== Derived (no need to touch) ==================

HOME_DIR = Path.home().resolve()
NVIDIA_API_KEY = (NVIDIA_API_KEY or os.environ.get("NVIDIA_API_KEY", "")).strip()
NGC_CLI_API_KEY = (NGC_CLI_API_KEY or os.environ.get("NGC_CLI_API_KEY", "")).strip()
HARDWARE_PROFILE = (HARDWARE_PROFILE or os.environ.get("HARDWARE_PROFILE", "RTXPRO6000BW")).strip()
EXTERNAL_IP = (EXTERNAL_IP or os.environ.get("EXTERNAL_IP", "")).strip()
if not EXTERNAL_IP:
    try:
        EXTERNAL_IP = subprocess.check_output(["hostname", "-I"], text=True).split()[0]
    except (subprocess.SubprocessError, IndexError):
        EXTERNAL_IP = ""
VSS_REPO_DIR = Path(os.environ.get("VSS_REPO_DIR", HOME_DIR / "video-search-and-summarization")).resolve()
NEMOCLAW_REPO_DIR = Path(os.environ.get("NEMOCLAW_REPO_DIR", HOME_DIR / "NemoClaw")).resolve()
DEPLOY_SCRIPTS_DIR = VSS_REPO_DIR / "deploy" / "docker" / "scripts"
NEMOHERMES_INIT_SCRIPT_PATH = DEPLOY_SCRIPTS_DIR / "nemohermes" / "init_nemohermes.sh"
NEMOCLAW_PROVIDER = "custom" if NEMOCLAW_ENDPOINT_URL else "build"
POLICY_PATH = VSS_REPO_DIR / "assets" / "vss_nemoclaw_policy.yaml"
SKILLS_DIR = VSS_REPO_DIR / "skills"
NEMOHERMES_WORKSPACE_DIR = VSS_REPO_DIR / ".hermes" / "workspace"
AGENT_DIR = VSS_REPO_DIR / "services" / "agent"
MCP_CONFIG_PATH = DEPLOY_SCRIPTS_DIR / "vss_orchestrator_mcp_config.yml"
ORCHESTRATOR_MCP_HELPER_PATH = DEPLOY_SCRIPTS_DIR / "orchestrator_mcp_helper.py"
ARTIFACT_DIR = VSS_REPO_DIR / ".orchestrator-artifacts"
LOG_PATH = ARTIFACT_DIR / "vss_orchestrator_mcp.log"
UV_BIN_DIR = HOME_DIR / ".local" / "bin" / "uv"
LLM_DEVICE_ID = str(LLM_DEVICE_ID or os.environ.get("LLM_DEVICE_ID", "")).strip()
VLM_DEVICE_ID = str(VLM_DEVICE_ID or os.environ.get("VLM_DEVICE_ID", "")).strip()
MCP_HOST = os.environ.get("VSS_ORCHESTRATOR_MCP_HOST", "0.0.0.0").strip()
MCP_PORT = int(os.environ.get("VSS_ORCHESTRATOR_MCP_PORT", "9988"))
HOST_INTERNAL_ALIAS = "host.openshell.internal"
MCP_URL = f"http://{MCP_HOST}:{MCP_PORT}/mcp"
SANDBOX_MCP_URL = f"http://{HOST_INTERNAL_ALIAS}:{MCP_PORT}/mcp"
NEMOCLAW_SANDBOX_NAME = os.environ.get("NEMOCLAW_SANDBOX_NAME", "vss-hermes").strip()
NEMOCLAW_INSTALL_REF = os.environ.get("NEMOCLAW_INSTALL_REF", NEMOCLAW_INSTALL_REF).strip()
NEMOHERMES_API_PORT = int(os.environ.get("NEMOHERMES_API_PORT", str(NEMOHERMES_API_PORT)))
NEMOCLAW_HERMES_DASHBOARD = str(os.environ.get("NEMOCLAW_HERMES_DASHBOARD", str(NEMOCLAW_HERMES_DASHBOARD))).strip().lower() in {"1", "true", "yes", "on"}
NEMOCLAW_HERMES_DASHBOARD_PORT = int(os.environ.get("NEMOCLAW_HERMES_DASHBOARD_PORT", str(NEMOCLAW_HERMES_DASHBOARD_PORT)))
TEST_PORTS = (3000, 8000, 9988, 30888, 5601, 6006, 9200, 8081, 31000, 9901, 38111, 38112)

print("Sandbox:", NEMOCLAW_SANDBOX_NAME)
print("NEMOCLAW_INSTALL_REF:", NEMOCLAW_INSTALL_REF)
print("\nHOME_DIR:", HOME_DIR)
print("VSS_REPO_DIR:", VSS_REPO_DIR)
print("NEMOHERMES_INIT_SCRIPT_PATH:", NEMOHERMES_INIT_SCRIPT_PATH)
print("NEMOCLAW_REPO_DIR:", NEMOCLAW_REPO_DIR)
print("POLICY_PATH:", POLICY_PATH)
print("SKILLS_DIR:", SKILLS_DIR)
print("NEMOHERMES_WORKSPACE_DIR:", NEMOHERMES_WORKSPACE_DIR)
print("AGENT_DIR:", AGENT_DIR)
print("MCP_CONFIG_PATH:", MCP_CONFIG_PATH)
print("ORCHESTRATOR_MCP_HELPER_PATH:", ORCHESTRATOR_MCP_HELPER_PATH)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("LOG_PATH:", LOG_PATH)
print("HARDWARE_PROFILE:", HARDWARE_PROFILE)
print("EXTERNAL_IP:", EXTERNAL_IP or "(unresolved)")
print("LLM_DEVICE_ID:", LLM_DEVICE_ID or "(profile default)")
print("VLM_DEVICE_ID:", VLM_DEVICE_ID or "(profile default)")
print("HOST_INTERNAL_ALIAS:", HOST_INTERNAL_ALIAS)
print("MCP_URL:", MCP_URL)
print("SANDBOX_MCP_URL:", SANDBOX_MCP_URL)
print("NEMOCLAW_PROVIDER:", NEMOCLAW_PROVIDER)
print("NEMOCLAW_MODEL:", NEMOCLAW_MODEL or "(init script default)")
print("Hermes API:", f"http://127.0.0.1:{NEMOHERMES_API_PORT}/v1")
print("Hermes dashboard:", f"http://127.0.0.1:{NEMOCLAW_HERMES_DASHBOARD_PORT}/" if NEMOCLAW_HERMES_DASHBOARD else "disabled")
if NEMOCLAW_ENDPOINT_URL:
    print("NEMOCLAW_ENDPOINT_URL:", NEMOCLAW_ENDPOINT_URL)


## 2. Preflight

Run the next cell to confirm the expected keys, files, commands are present on the instance.


In [ ]:
import ast
import importlib.util
import os
import shutil
from pathlib import Path

RED = "\033[31m"
RESET = "\033[0m"
GREEN = "\033[32m"
YELLOW = "\033[33m"

helper_spec = importlib.util.spec_from_file_location("orchestrator_mcp_helper", ORCHESTRATOR_MCP_HELPER_PATH)
if helper_spec is None or helper_spec.loader is None:
    raise ImportError(f"Could not load MCP helper from {ORCHESTRATOR_MCP_HELPER_PATH}")
orchestrator_mcp_helper = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(orchestrator_mcp_helper)

OrchestratorTool = orchestrator_mcp_helper.OrchestratorTool
build_vss_ui_url = orchestrator_mcp_helper.build_vss_ui_url
poll_compose_op = orchestrator_mcp_helper.poll_compose_op
require_success = orchestrator_mcp_helper.require_success
tool_call = orchestrator_mcp_helper.tool_call

if not shutil.which("uv") and UV_BIN_DIR.exists():
    os.environ["PATH"] = f"{UV_BIN_DIR.parent}:{os.environ.get('PATH', '')}"

required_checks = {
    "init_nemohermes.sh": NEMOHERMES_INIT_SCRIPT_PATH.is_file(),
    "vss_nemoclaw_policy.yaml": POLICY_PATH.is_file(),
    "skills/": SKILLS_DIR.is_dir(),
    "Hermes workspace templates": NEMOHERMES_WORKSPACE_DIR.is_dir(),
    "services/agent/": AGENT_DIR.is_dir(),
    "vss_orchestrator_mcp_config.yml": MCP_CONFIG_PATH.is_file(),
    "orchestrator_mcp_helper.py": ORCHESTRATOR_MCP_HELPER_PATH.is_file(),
    "docker": shutil.which("docker") is not None,
    "python3": shutil.which("python3") is not None,
    "curl": shutil.which("curl") is not None,
    "uv": shutil.which("uv") is not None,
    "NGC_CLI_API_KEY set": bool(NGC_CLI_API_KEY),
}

for label, ok in required_checks.items():
    status = "OK " if ok else "NO "
    color = GREEN if ok else RED
    print(f"{color}{status}{RESET} {label}")

if NEMOCLAW_PROVIDER == "build" and not NVIDIA_API_KEY:
    print(f"{RED}NO {RESET} NVIDIA_API_KEY set for build provider")
if NEMOCLAW_PROVIDER == "custom" and (not NEMOCLAW_ENDPOINT_URL or not COMPATIBLE_API_KEY):
    print(f"{RED}NO {RESET} NEMOCLAW_ENDPOINT_URL / COMPATIBLE_API_KEY set for custom provider")

# Catch notebook code-cell generation mistakes before users hit them later.
nb_path = DEPLOY_SCRIPTS_DIR / "deploy_nemohermes_vss.ipynb"
if nb_path.is_file():
    import json
    nb = json.loads(nb_path.read_text())
    for i, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        if source.lstrip().startswith("%%"):
            continue
        ast.parse(source)
    print(f"{GREEN}OK {RESET} notebook code-cell syntax")


### 2.1 Pin Docker version

Pin Docker CE + plugins + containerd.io to a known-good combination (CE **29.4.3**, buildx **0.33.0**, compose **5.1.3**, containerd **2.2.3**) **before** section 3 brings up the NemoHermes sandbox — a docker-ce downgrade restarts dockerd and would disrupt live sandbox containers if it ran later in the notebook. `apt-mark hold` prevents drift back to newer versions before section 6 runs.

The cell first reads the installed Docker Engine version: if it already falls in the tested range **[28.3.3, 29.5.0)** the version downgrade is **skipped**. Safe to re-run.


In [ ]:
%%bash
# Pin Docker CE + plugins + containerd.io to a known-good combination, but
# ONLY when the host's Docker is outside the tested range. Some Brev
# launchables ship a newer Docker than the VSS deploy profiles are tested
# against; pin explicitly so compose/buildx incompatibilities don't surface
# mid-deployment.
#
# When the installed Docker already falls in [28.3.3, 29.5.0) the version
# downgrade is skipped: re-pinning to an exact epoch-versioned package that
# the platform's apt repo may not carry (e.g. DGX Spark / DGX-OS on arm64)
# fails with "version not found" for no benefit. The in-range packages are
# still held so the box can't drift past the tested range mid-notebook.
# Idempotent -- safe to re-run.

set -euo pipefail

# Tested Docker Engine range -- keep in sync with the VSS launchable prereq check.
MIN_DOCKER_VERSION="28.3.3"
MAX_DOCKER_VERSION="29.5.0"

# Packages frozen with `apt-mark hold` so unattended-upgrades / later
# `apt-get install` calls can't drift the box before the notebook finishes.
HOLD_PKGS="docker-ce docker-ce-cli docker-buildx-plugin docker-compose-plugin containerd.io"

version_ge() { [ "$(printf '%s\n%s\n' "$2" "$1" | sort -V | head -n1)" = "$2" ]; }
version_lt() { [ "$1" != "$2" ] && [ "$(printf '%s\n%s\n' "$1" "$2" | sort -V | head -n1)" = "$1" ]; }

DOCKER_VERSION="$(docker version --format '{{.Server.Version}}' 2>/dev/null || true)"
if [ -n "$DOCKER_VERSION" ] \
   && version_ge "$DOCKER_VERSION" "$MIN_DOCKER_VERSION" \
   && version_lt "$DOCKER_VERSION" "$MAX_DOCKER_VERSION"; then
  echo "Docker $DOCKER_VERSION is within the tested range [$MIN_DOCKER_VERSION, $MAX_DOCKER_VERSION); skipping the Docker version pin."
  # No downgrade needed, but still hold the in-range packages at their
  # current versions so unattended-upgrades / later apt-get calls can't drift
  # the box past the tested range for the remainder of the notebook.
  sudo apt-mark hold $HOLD_PKGS
  exit 0
fi

if [ -n "$DOCKER_VERSION" ]; then
  echo "Docker $DOCKER_VERSION is outside the tested range [$MIN_DOCKER_VERSION, $MAX_DOCKER_VERSION); pinning to known-good versions."
else
  echo "Could not read the installed Docker version; pinning to known-good versions."
fi

# Read distro info from /etc/os-release (always present on Ubuntu; minimal
# images don't ship `lsb_release`).
. /etc/os-release
DISTRO="${VERSION_ID}"
CODENAME="${UBUNTU_CODENAME:-${VERSION_CODENAME}}"

# Versions hard-coded to what shipped alongside docker-ce 29.4.3 on the
# Docker apt repo (verified against download.docker.com + upstream GitHub
# release timestamps). When bumping DOCKER_CE_VER, bump these four together.
DOCKER_CE_VER="5:29.4.3-1~ubuntu.${DISTRO}~${CODENAME}"
BUILDX_VER="0.33.0-1~ubuntu.${DISTRO}~${CODENAME}"
COMPOSE_VER="5.1.3-1~ubuntu.${DISTRO}~${CODENAME}"
CONTAINERD_VER="2.2.3-1~ubuntu.${DISTRO}~${CODENAME}"

# Refresh the APT cache first -- without this, the specific epoch-versioned
# package may not be in the local index and the install would fail with
# version-not-found before any pinning takes effect.
sudo apt-get update -qq

sudo DEBIAN_FRONTEND=noninteractive apt-get install -y \
  --allow-downgrades \
  -o Dpkg::Options::=--force-confdef \
  -o Dpkg::Options::=--force-confold \
  docker-ce="$DOCKER_CE_VER" \
  docker-ce-cli="$DOCKER_CE_VER" \
  docker-buildx-plugin="$BUILDX_VER" \
  docker-compose-plugin="$COMPOSE_VER" \
  containerd.io="$CONTAINERD_VER"

# Hold so unattended-upgrades / later `apt-get install` calls don't drift
# the box back to newer versions before the rest of the notebook runs.
sudo apt-mark hold $HOLD_PKGS

## 3. Install and Configure NemoHermes for VSS skills

The next cell first runs the pinned NemoClaw installer from `NEMOCLAW_INSTALL_REF`, then runs the VSS NemoHermes init script in the foreground so you can watch progress.

The installer/init flow will:

- create or update the NemoClaw sandbox with the Hermes agent,
- configure the OpenShell provider for the selected model provider,
- apply the VSS NemoClaw policy,
- install VSS skills,
- upload Hermes workspace instructions,
- configure NGC credentials when available,
- verify the Hermes API health endpoint.


In [ ]:
import os
import re
import shlex
import shutil
import shutil
import socket
import subprocess
import time
import urllib.request
from collections import deque
from pathlib import Path

if not NEMOHERMES_INIT_SCRIPT_PATH.is_file():
    raise FileNotFoundError(f"Missing init script: {NEMOHERMES_INIT_SCRIPT_PATH}")

if NEMOCLAW_PROVIDER == "build" and not NVIDIA_API_KEY:
    raise RuntimeError(
        "No Hermes provider configured. Pick one and fill in its values:\n"
        "  - option (a) SOTA cloud / (b) local: set NEMOCLAW_ENDPOINT_URL, NEMOCLAW_MODEL, and COMPATIBLE_API_KEY in the (a) or (b) cell\n"
        "  - option (c) build.nvidia.com: set NVIDIA_API_KEY in the (c) cell"
    )
if NEMOCLAW_PROVIDER == "custom" and not COMPATIBLE_API_KEY:
    raise RuntimeError(
        "COMPATIBLE_API_KEY is required for the custom OpenAI-compatible provider (options a/b). "
        "Set it in the (a) or (b) cell (any non-empty placeholder works for most local servers)."
    )
if NEMOCLAW_PROVIDER == "custom" and not NEMOCLAW_MODEL:
    raise RuntimeError(
        "NEMOCLAW_MODEL is required when NEMOCLAW_ENDPOINT_URL is set. "
        "If you ran more than one of (a)/(b)/(c), re-run just the (a) or (b) cell you want to use."
    )

if not POLICY_PATH.is_file():
    raise FileNotFoundError(f"Missing policy file: {POLICY_PATH}")
if not SKILLS_DIR.is_dir():
    raise FileNotFoundError(f"Missing skills dir: {SKILLS_DIR}")
if not NEMOHERMES_WORKSPACE_DIR.is_dir():
    raise FileNotFoundError(f"Missing NemoHermes workspace dir: {NEMOHERMES_WORKSPACE_DIR}")

env = os.environ.copy()
env["VSS_REPO_DIR"] = str(VSS_REPO_DIR)
env["NEMOCLAW_REPO_DIR"] = str(NEMOCLAW_REPO_DIR)
env["NEMOCLAW_SANDBOX_NAME"] = NEMOCLAW_SANDBOX_NAME
env["NEMOCLAW_AGENT"] = "hermes"
env["NEMOCLAW_INSTALL_REF"] = NEMOCLAW_INSTALL_REF
env["NVIDIA_API_KEY"] = NVIDIA_API_KEY
env["NGC_CLI_API_KEY"] = NGC_CLI_API_KEY
env["NEMOCLAW_PROVIDER"] = NEMOCLAW_PROVIDER
env["NEMOCLAW_NON_INTERACTIVE"] = "1"
env["NEMOCLAW_ACCEPT_THIRD_PARTY_SOFTWARE"] = "1"
env["NEMOCLAW_POLICY_FILE"] = str(POLICY_PATH)
env["NEMOHERMES_WORKSPACE_DIR"] = str(NEMOHERMES_WORKSPACE_DIR)
env["NEMOHERMES_API_PORT"] = str(NEMOHERMES_API_PORT)
env["NEMOHERMES_MCP_URL"] = SANDBOX_MCP_URL
env["NEMOCLAW_HERMES_DASHBOARD"] = "1" if NEMOCLAW_HERMES_DASHBOARD else "0"
env["NEMOCLAW_HERMES_DASHBOARD_PORT"] = str(NEMOCLAW_HERMES_DASHBOARD_PORT)

if NEMOCLAW_MODEL:
    env["NEMOCLAW_MODEL"] = NEMOCLAW_MODEL

if NEMOCLAW_PROVIDER == "custom":
    env["NEMOCLAW_ENDPOINT_URL"] = NEMOCLAW_ENDPOINT_URL
    env["COMPATIBLE_API_KEY"] = COMPATIBLE_API_KEY
    print(f"[{HARDWARE_PROFILE}] OpenAI-compatible provider: {NEMOCLAW_ENDPOINT_URL} model={NEMOCLAW_MODEL} key set={bool(COMPATIBLE_API_KEY)}")

timeout_sec = 3600


def expected_nemoclaw_version(ref):
    match = re.fullmatch(r"v?(\d+\.\d+\.\d+)", ref.strip())
    return match.group(1) if match else None


def installed_nemoclaw_version():
    nemoclaw_bin = shutil.which("nemoclaw")
    if nemoclaw_bin is None:
        return None
    try:
        output = subprocess.check_output(
            [nemoclaw_bin, "--version"],
            text=True,
            stderr=subprocess.STDOUT,
            timeout=30,
        ).strip()
    except (subprocess.SubprocessError, OSError):
        return None
    match = re.search(r"v?(\d+\.\d+\.\d+)", output)
    return match.group(1) if match else output


def run_streamed(command, *, label, cwd):
    last_output_lines = deque(maxlen=40)
    start = time.monotonic()
    print("Running:", label, flush=True)
    process = subprocess.Popen(
        command,
        env=env,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    try:
        if process.stdout is None:
            raise RuntimeError(f"Failed to capture {label} output")

        for line in process.stdout:
            print(line, end="", flush=True)
            last_output_lines.append(line.rstrip("\n"))
            if time.monotonic() - start > timeout_sec:
                process.kill()
                process.wait()
                raise TimeoutError(f"{label} timed out after {timeout_sec} seconds")
    finally:
        if process.stdout is not None:
            process.stdout.close()

    returncode = process.wait()
    if returncode != 0:
        tail = "\n".join(last_output_lines)
        raise RuntimeError(
            f"{label} exited with {returncode}\n"
            f"Last output:\n{tail}"
        )


def wait_for_hermes_health(timeout_s=120):
    health_url = f"http://127.0.0.1:{NEMOHERMES_API_PORT}/health"
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urllib.request.urlopen(health_url, timeout=5) as resp:
                body = resp.read().decode("utf-8", errors="replace")[:500]
                print("Hermes health:", resp.status, body)
                return
        except Exception as exc:
            last_error = str(exc)
            time.sleep(3)
    raise RuntimeError(f"Hermes API did not become healthy at {health_url}: {last_error}")


install_ref = NEMOCLAW_INSTALL_REF.strip()
if not install_ref:
    raise RuntimeError("NEMOCLAW_INSTALL_REF must not be empty.")
install_url = f"https://raw.githubusercontent.com/NVIDIA/NemoClaw/{install_ref}/install.sh"
installer_cmd = f"curl -fsSL {shlex.quote(install_url)} | bash"

print("Using NemoClaw install ref:", install_ref, flush=True)
print("Using NemoClaw repo:", NEMOCLAW_REPO_DIR, flush=True)
print("Applying policy file:", POLICY_PATH, flush=True)
print("Using NemoHermes init script:", NEMOHERMES_INIT_SCRIPT_PATH, flush=True)

installed_version = installed_nemoclaw_version()
expected_version = expected_nemoclaw_version(install_ref)
should_run_installer = installed_version is None or expected_version is None or installed_version != expected_version
if should_run_installer:
    if installed_version and expected_version is None:
        print(f"NEMOCLAW_INSTALL_REF {install_ref!r} is not a semver tag; reinstalling to honor the requested ref.", flush=True)
    elif installed_version:
        print(f"NemoClaw installed version {installed_version} does not match {install_ref}; reinstalling.", flush=True)
    run_streamed(["bash", "-lc", installer_cmd], label="NemoClaw installer", cwd=str(HOME_DIR))
else:
    print(f"NemoClaw {installed_version} already installed, skipping installer.", flush=True)
run_streamed(["bash", str(NEMOHERMES_INIT_SCRIPT_PATH)], label="init_nemohermes.sh", cwd=str(NEMOHERMES_INIT_SCRIPT_PATH.parent))
wait_for_hermes_health()
print("Done.", flush=True)


## 4. [OPTIONAL] Verify sandbox, policy, skills, and Hermes workspace

The next cell checks:

- whether the sandbox exists,
- the current active sandbox policy metadata,
- the expected local policy path and preset name,
- whether the Hermes CLI is available in the sandbox,
- whether the NemoHermes workspace files were uploaded,
- whether NemoClaw can list installed skills.


In [ ]:
from datetime import datetime, timezone
import json
import re
import shlex
import shutil
import subprocess
from pathlib import Path


def run(cmd, check=False, echo=True):
    print("$", shlex.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True, check=check)
    if echo:
        if r.stdout:
            print(r.stdout)
        if r.stderr:
            print(r.stderr)
    return r


print("Verification time (UTC):", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S %Z"))
print("Sandbox:", NEMOCLAW_SANDBOX_NAME)
print("Expected policy file:", POLICY_PATH)

policy_text = Path(POLICY_PATH).read_text()
preset_name_match = re.search(r"^\s+name:\s*(\S+)", policy_text, re.MULTILINE)
print("Expected preset name:", preset_name_match.group(1) if preset_name_match else "unknown")

sandbox_result = run(["openshell", "sandbox", "get", NEMOCLAW_SANDBOX_NAME], echo=False)
sandbox_summary = "\n".join(part for part in (sandbox_result.stdout, sandbox_result.stderr) if part)
sandbox_summary = re.sub(r"\x1b\[[0-9;]*m", "", sandbox_summary)
phase_match = re.search(r"Phase:\s+(.+)", sandbox_summary)
namespace_match = re.search(r"Namespace:\s+(.+)", sandbox_summary)
sandbox_id_match = re.search(r"Id:\s+(.+)", sandbox_summary)
print("Sandbox namespace:", namespace_match.group(1).strip() if namespace_match else "unknown")
print("Sandbox phase:", phase_match.group(1).strip() if phase_match else "unknown")
print("Sandbox id:", sandbox_id_match.group(1).strip() if sandbox_id_match else "unknown")

policy_result = run(["openshell", "policy", "get", NEMOCLAW_SANDBOX_NAME])
policy_summary = "\n".join(part for part in (policy_result.stdout, policy_result.stderr) if part)
status_match = re.search(r"Status:\s+(.+)", policy_summary)
active_match = re.search(r"Active:\s+(.+)", policy_summary)
hash_match = re.search(r"Hash:\s+([0-9a-f]+)", policy_summary)
print("Active policy status:", status_match.group(1).strip() if status_match else "unknown")
print("Active policy version:", active_match.group(1).strip() if active_match else "unknown")
print("Active policy hash:", hash_match.group(1) if hash_match else "unknown")

print("\n=== Hermes CLI ===")
run(["openshell", "sandbox", "exec", "-n", NEMOCLAW_SANDBOX_NAME, "--", "sh", "-lc", "command -v hermes && hermes --version || true"])

print("\n=== NemoHermes workspace files ===")
run([
    "openshell", "sandbox", "exec", "-n", NEMOCLAW_SANDBOX_NAME, "--",
    "sh", "-lc", "ls -1 /sandbox/.hermes-data/workspace 2>/dev/null | sed 's/^/- /'",
])

print("\n=== Installed skills (NemoHermes) ===")
skill_cli = shutil.which("nemohermes") or shutil.which("nemoclaw")
if skill_cli is None:
    raise RuntimeError("Neither nemohermes nor nemoclaw is available for skill listing")
skills_result = run([skill_cli, NEMOCLAW_SANDBOX_NAME, "skill", "list"], echo=True)
if skills_result.returncode != 0:
    print("Skill list failed; this may indicate an older NemoClaw CLI. Check the init_nemohermes.sh install logs.")

print("\n=== Hermes API health ===")
run(["curl", "-fsS", f"http://127.0.0.1:{NEMOHERMES_API_PORT}/health"])


## 5. Prepare the host for local NIM-backed VSS profiles

If you plan to deploy local NIM-backed VSS profiles through the orchestrator tools, complete the next three pre-steps on the host first:

1. install and configure the NGC CLI,
2. authenticate Docker to `nvcr.io`, and
3. prepare the `services/agent/` Python environment.

### 5.1 Install and configure NGC CLI

This pre-step checks whether `ngc` is already installed, installs it if needed, and writes the local NGC CLI config using `NGC_CLI_API_KEY`.


In [ ]:
import os
import platform
import shutil
import shutil
import subprocess


def run(cmd: str) -> str:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{result.stderr}\n{result.stdout}")
    return result.stdout.strip()


ngc_path = shutil.which("ngc")
if ngc_path:
    version = run("ngc --version 2>&1 | head -1")
    print(f"NGC CLI already installed: {version}")
else:
    arch = platform.machine()
    filename = "ngccli_arm64.zip" if arch in ("aarch64", "arm64") else "ngccli_linux.zip"
    ngc_cli_version = "4.13.0"
    url = (
        "https://api.ngc.nvidia.com/v2/resources/nvidia/ngc-apps/ngc_cli/"
        f"versions/{ngc_cli_version}/files/{filename}"
    )

    print(f"Installing NGC CLI {ngc_cli_version} ({filename})...")
    run(f"cd /tmp && curl -fL --retry 3 --retry-delay 2 -o ngc_cli.zip '{url}'")

    size = os.path.getsize("/tmp/ngc_cli.zip")
    if size < 1000:
        raise RuntimeError(
            f"NGC CLI download failed: /tmp/ngc_cli.zip is only {size} bytes."
        )

    run("cd /tmp && unzip -o ngc_cli.zip")
    run("sudo cp -r /tmp/ngc-cli/* /usr/local/bin/")
    run("rm -rf /tmp/ngc_cli.zip /tmp/ngc-cli")

    version = run("ngc --version 2>&1 | head -1")
    print(f"Installed NGC CLI: {version}")


if not NGC_CLI_API_KEY:
    raise RuntimeError(
        "NGC_CLI_API_KEY is not set. Set it in the notebook settings cell or export it "
        "in the environment before running this step."
    )

print("Configuring NGC CLI...")
ngc_dir = os.path.expanduser("~/.ngc")
os.makedirs(ngc_dir, exist_ok=True)

with open(os.path.join(ngc_dir, "config"), "w") as f:
    f.write(f""";WARNING - This is a machine generated file. Do not edit manually.
;WARNING - To update local config settings, see 'ngc config set -h'.

[CURRENT]
apikey = {NGC_CLI_API_KEY}
format_type = ascii
org = nvstaging
""")

print("NGC CLI configured.")
print(run("ngc config current"))

### 5.2 Docker login to `nvcr.io`

If you plan to deploy local NIM-backed VSS profiles, authenticate Docker to the NVIDIA Container Registry before using the orchestrator deployment tools.

This pre-step runs `docker login nvcr.io` with `NGC_CLI_API_KEY`.


In [ ]:
import subprocess

if not NGC_CLI_API_KEY:
    raise RuntimeError("NGC_CLI_API_KEY is not set. Export it before running this cell.")

login_result = subprocess.run(
    [
        "docker",
        "login",
        "nvcr.io",
        "--username",
        "$oauthtoken",
        "--password",
        NGC_CLI_API_KEY,
    ],
    capture_output=True,
    text=True,
)
if login_result.returncode != 0:
    raise RuntimeError(f"Docker login to nvcr.io failed\n{login_result.stderr}")

print("Docker login to nvcr.io: OK")

### 5.3 Prepare the `services/agent/` environment

The orchestrator MCP server is part of the VSS agent package, so install the Python environment in `services/agent/` before starting the server.

This runs `uv sync` from the agent directory.


In [ ]:
import os
import shutil
import subprocess

if shutil.which("uv") is None:
    raise RuntimeError("uv is not installed. Install uv first, then re-run this cell.")

uv_env = os.environ.copy()
uv_env.pop("VIRTUAL_ENV", None)
subprocess.run(["uv", "sync"], cwd=str(AGENT_DIR), env=uv_env, check=True)
venv_info = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "-c",
        "import os, sys; print('uv VIRTUAL_ENV:', os.environ.get('VIRTUAL_ENV'))",
    ],
    cwd=str(AGENT_DIR),
    env=uv_env,
    check=True,
    capture_output=True,
    text=True,
)
print(venv_info.stdout.strip())
print("Environment is ready.")

## 6. Start the VSS Orchestrator MCP server and deploy a profile

From here on, the agent — not this notebook — drives the VSS deployment. Work through the sub-steps in this order:

1. **6.1** — start the host-side VSS Orchestrator MCP server (a host process listening on port `9988`).
2. **6.3** — connect to NemoHermes, then run the smoke tests to confirm the agent can chat, sees the VSS skills, and can reach the orchestrator MCP server.
3. **6.2** — once the smoke tests pass, ask the agent to use the VSS Orchestrator MCP tools to deploy and manage VSS (including teardown via `docker_down`).

**6.2** is reference material (tool list + sample prompts) and has no code cell of its own — all the work happens in NemoHermes once **6.1** and **6.3** are done.


### 6.1 Start the VSS Orchestrator MCP server

Run the next cell to stop any previously recorded MCP server from this notebook session and start a fresh one.


In [ ]:
import importlib.util
import os
import shutil
import signal
import subprocess
import sys
import time


helper_spec = importlib.util.spec_from_file_location("orchestrator_mcp_helper", ORCHESTRATOR_MCP_HELPER_PATH)
if helper_spec is None or helper_spec.loader is None:
    raise ImportError(f"Could not load MCP helper from {ORCHESTRATOR_MCP_HELPER_PATH}")
orchestrator_mcp_helper = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(orchestrator_mcp_helper)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

for label, ok in {
    "uv": shutil.which("uv") is not None,
    "docker": shutil.which("docker") is not None,
    "openshell": shutil.which("openshell") is not None,
    "agent dir": AGENT_DIR.is_dir(),
    "MCP config": MCP_CONFIG_PATH.is_file(),
}.items():
    if not ok:
        raise RuntimeError(f"Cannot start the MCP server because {label} is unavailable. Resolve this before proceeding.")


def _wait_for_mcp_health(process: subprocess.Popen, timeout_s: int = 60, interval_s: int = 3) -> None:
    deadline = time.time() + timeout_s
    last_error = "health check did not run"
    health_url = f"http://127.0.0.1:{MCP_PORT}/mcp"
    while time.time() < deadline:
        return_code = process.poll()
        if return_code is not None:
            raise RuntimeError(f"MCP server exited before becoming healthy with exit code {return_code}: {last_error}")
        try:
            healthy, message = orchestrator_mcp_helper.check_mcp_health(health_url, AGENT_DIR)
        except subprocess.TimeoutExpired:
            healthy, message = False, "health command timed out"
        last_error = message
        if healthy:
            print(f"MCP health check passed: {message}")
            return
        time.sleep(interval_s)

    raise RuntimeError(f"MCP server did not become healthy within {timeout_s}s: {last_error}")


existing_pid = globals().get("VSS_ORCHESTRATOR_MCP_PID")
if existing_pid:
    try:
        os.kill(existing_pid, signal.SIGTERM)
        print(f"Stopped existing MCP server PID {existing_pid}")
        time.sleep(2)
    except ProcessLookupError:
        print(f"Recorded MCP server PID {existing_pid} is no longer running")

env = os.environ.copy()
env.setdefault("PYTHONUNBUFFERED", "1")
if NGC_CLI_API_KEY:
    env["NGC_CLI_API_KEY"] = NGC_CLI_API_KEY
if NVIDIA_API_KEY:
    env["NVIDIA_API_KEY"] = NVIDIA_API_KEY
if HARDWARE_PROFILE:
    env["HARDWARE_PROFILE"] = HARDWARE_PROFILE
env["EXTERNAL_IP"] = EXTERNAL_IP
if VSS_LLM_ENABLE_THINKING:
    env["LLM_ENABLE_THINKING"] = VSS_LLM_ENABLE_THINKING
if LLM_DEVICE_ID:
    env["LLM_DEVICE_ID"] = LLM_DEVICE_ID
if VLM_DEVICE_ID:
    env["VLM_DEVICE_ID"] = VLM_DEVICE_ID
if VSS_LLM_ENDPOINT_URL or VSS_VLM_ENDPOINT_URL:
    env["OPENAI_API_KEY"] = VSS_OPENAI_API_KEY or NVIDIA_API_KEY
if VSS_LLM_ENDPOINT_URL:
    env["LLM_ENDPOINT_URL"] = VSS_LLM_ENDPOINT_URL
    env["LLM_NAME"] = VSS_LLM_NAME
    env["LLM_MODEL_TYPE"] = VSS_LLM_MODEL_TYPE
if VSS_VLM_ENDPOINT_URL:
    env["VLM_NAME"] = VSS_VLM_NAME
    env["VLM_ENDPOINT_URL"] = VSS_VLM_ENDPOINT_URL
    env["VLM_MODEL_TYPE"] = VSS_VLM_MODEL_TYPE
log_handle = LOG_PATH.open("w")
process = subprocess.Popen(
    [
        "uv",
        "run",
        "nat",
        "mcp",
        "serve",
        "--config_file",
        str(MCP_CONFIG_PATH),
        "--host",
        MCP_HOST,
        "--port",
        str(MCP_PORT),
    ],
    cwd=str(AGENT_DIR),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,
)
VSS_ORCHESTRATOR_MCP_PID = process.pid
print(f"Started MCP server with PID {VSS_ORCHESTRATOR_MCP_PID}")
_wait_for_mcp_health(process)
print("MCP log:", LOG_PATH)
print("MCP URL:", MCP_URL)
print("Sandbox MCP URL:", SANDBOX_MCP_URL)
print("If NemoHermes is already connected, run /reload-mcp in the Hermes session or reconnect after this cell.")


### 6.2 Deploy VSS from NemoHermes via MCP tools

With the MCP server running, the **Hermes agent inside NemoClaw/OpenShell** can now drive the VSS deployment for you through chat. Connect to NemoHermes (use **6.3** below), run `/reload-mcp` if the session was already open, then ask the agent to use the orchestrator tools.

#### Available VSS Orchestrator MCP tools

The host MCP server is registered in Hermes config as `vss_orchestrator`. Hermes may display these tools with a client-specific prefix, but the underlying operations are:

| Tool | Purpose |
| --- | --- |
| `profiles` | List supported deployment profiles (`base`, `search`, `alerts`, `lvs`). |
| `prereqs` | Run Docker / GPU / NGC prerequisite checks on the host. |
| `docker_generate` | Resolve `.env` + compose YAML artifacts for the chosen profile. |
| `docker_read` | Fetch generated env/yaml by `docker_compose_id`. |
| `docker_up` | `docker compose up -d --build --quiet-pull` for the generated artifacts. |
| `docker_status` | Poll status/logs of the most recent `docker_up` / `docker_down` operation. |
| `docker_list` | List currently running container names. |
| `docker_logs` | Fetch docker logs for a given container name. |
| `docker_down` | `docker compose down -v --remove-orphans` to tear the deployment back down. |

#### Sample prompts to trigger them

Try prompts like:

| Sample prompt | Tool(s) invoked |
| --- | --- |
| *"List the available VSS deployment profiles."* | `profiles` |
| *"Check that my host meets the prerequisites for the `alerts` profile."* | `prereqs` |
| *"Generate VSS base deployment artifacts and stop before docker_up."* | `docker_generate` |
| *"Deploy the VSS `alerts` profile in `verification` mode."* | `docker_generate` -> `docker_up` -> `docker_status` |
| *"List the running VSS containers."* | `docker_list` |
| *"Fetch the last 200 lines of logs from `vss-alert-bridge`."* | `docker_logs` |
| *"Tear down the VSS deployment."* | `docker_down` |

The agent should ask before destructive steps and stream progress back into the chat. If a tool call fails, paste the error message back to the agent and ask it to remediate — it has access to `docker_logs` and `docker_status` to diagnose.


### 6.3 Connect to NemoHermes and verify it

Run the next cell to print connection/API details.

<span style="color:red"><strong>Not on Brev?</strong> If you are accessing the Hermes API from a different machine, open an SSH tunnel before calling the API from your laptop:<br/><code>ssh -L 8642:127.0.0.1:8642 &lt;user&gt;@&lt;nemoclaw-host&gt;</code><br/>If you enabled the Hermes dashboard, also forward <code>9119</code>.</span>


In [ ]:
import shutil
import subprocess
import urllib.request

print("Connect command:")
print(f"  nemohermes {NEMOCLAW_SANDBOX_NAME} connect")
print("\nHermes API:")
print(f"  http://127.0.0.1:{NEMOHERMES_API_PORT}/v1")
if NEMOCLAW_HERMES_DASHBOARD:
    print("\nHermes dashboard:")
    print(f"  http://127.0.0.1:{NEMOCLAW_HERMES_DASHBOARD_PORT}/")
print("\nMCP from sandbox:")
print(f"  {SANDBOX_MCP_URL}")
print("\nMCP reload:")
print("  If the MCP server was started after NemoHermes connected, run /reload-mcp in Hermes or reconnect.")

health_url = f"http://127.0.0.1:{NEMOHERMES_API_PORT}/health"
try:
    with urllib.request.urlopen(health_url, timeout=5) as resp:
        print("\nHermes health:", resp.status, resp.read().decode("utf-8", errors="replace")[:500])
except Exception as exc:
    raise RuntimeError(f"Hermes API health check failed at {health_url}: {exc}") from exc

print("\nInstalled skill list probe:")
skill_cli = shutil.which("nemohermes") or shutil.which("nemoclaw")
if skill_cli is None:
    raise RuntimeError("Neither nemohermes nor nemoclaw is available for skill listing")
skills = subprocess.run(
    [skill_cli, NEMOCLAW_SANDBOX_NAME, "skill", "list"],
    capture_output=True,
    text=True,
)
print(skills.stdout or skills.stderr)


#### Step 2. Verify the LLM works in chat

Connect with:

```bash
nemohermes vss-hermes connect
```

Use your sandbox name if you changed `NEMOCLAW_SANDBOX_NAME`.

Start with simple prompts:

- `hello`
- `what model are you using?`
- `list your available skills`

#### Step 3. Verify the VSS skills are imported

Ask:

> *"List your VSS skills."*

You should see the VSS skill set installed from this repository.

#### Step 4. Verify the agent can reach the orchestrator MCP server

Run the prompts below in order. If any step fails, re-check that **6.1** completed successfully and that the MCP server is still running on port `9988`.

**Prompt A — show deployment tools:**

> *"Show me the deployment tools."*

The agent should summarize the VSS Orchestrator MCP deployment tools.

**Prompt B — query a tool (list profiles):**

> *"List the available VSS deployment profiles."*

The agent should invoke the orchestrator `profiles` operation and return `base`, `search`, `alerts`, `lvs`.

**Prompt C — invoke a deployment:**

> *"Deploy the VSS `alerts` profile in `verification` mode."*

The agent should chain orchestrator `docker_generate` -> `docker_up` -> `docker_status` and stream progress back into the chat. It will ask before invoking the build, and surface a `docker_compose_id` you can reference later.


## 7. [OPTIONAL] Verify host reachability from inside the sandbox

<span style="color:red"><strong>Important:</strong> make sure VSS is already deployed on the host before running this step.</span>

Run the next cell only after the agent has finished deploying VSS in **section 6**. It runs `nemohermes <sandbox> connect` and feeds a probe script into that real sandbox session, which matches the NemoHermes path used by skills and tools. Do **not** replace this with `docker exec`; that can use a different path and produce misleading results.

The `STATUS` column is intentionally simple:

- `REACHABLE` — an HTTP service answered on that port. `404` and non-policy `403` still count because they prove a service is listening, even if `GET /` is not a valid or authorized route.
- `NOT_REACHABLE` — the port is blocked by policy, no service is listening, DNS failed, or the request timed out. Check the `NOTE` column for the reason.


In [ ]:
import re
import subprocess

required_vars = ("NEMOCLAW_SANDBOX_NAME", "HOST_INTERNAL_ALIAS", "TEST_PORTS")
missing_vars = [name for name in required_vars if name not in globals()]
if missing_vars:
    raise RuntimeError("Required variables are missing: " + ", ".join(missing_vars))

ports = " ".join(str(port) for port in TEST_PORTS)
probe = f"""
HOST_IP="${{HOST_IP:-{HOST_INTERNAL_ALIAS}}}"
PORTS="{ports}"

export HOST_IP

echo __VSS_PORT_PROBE_BEGIN__
for port in $PORTS; do
  body="$(mktemp)"
  http_code="$(curl -sS --max-time 5 -o "$body" -w '%{{http_code}}' "http://${{HOST_IP}}:${{port}}/" 2>/tmp/portcheck.err)"
  curl_exit=$?
  body_text="$(tr '[:upper:]' '[:lower:]' < "$body")"

  if printf '%s' "$body_text" | grep -Eq 'policy_denied|egress.*denied|not allowed by policy'; then
    status="NOT_REACHABLE"
    note="blocked by policy"
  elif [ "$http_code" = "502" ] || printf '%s' "$body_text" | grep -q 'upstream_unreachable'; then
    status="NOT_REACHABLE"
    note="allowed, no service listening"
  elif [ "$http_code" != "000" ]; then
    status="REACHABLE"
    note="HTTP $http_code response"
  else
    status="NOT_REACHABLE"
    case "$curl_exit" in
      7) note="connect failed" ;;
      28) note="timeout" ;;
      *) note="curl_exit=$curl_exit" ;;
    esac
  fi

  printf '%s|%s|%s|%s\n' "$port" "$status" "$http_code" "$note"
  rm -f "$body" /tmp/portcheck.err
done
echo __VSS_PORT_PROBE_END__
exit
""".strip() + "\n"

print(f"Running reachability probe through: nemohermes {NEMOCLAW_SANDBOX_NAME} connect")
result = subprocess.run(
    ["nemohermes", NEMOCLAW_SANDBOX_NAME, "connect"],
    input=probe,
    capture_output=True,
    text=True,
    timeout=120,
)

lines = result.stdout.splitlines()
ansi_escape = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")
try:
    begin = next(i for i, line in enumerate(lines) if line.strip() == "__VSS_PORT_PROBE_BEGIN__")
    end = next(i for i, line in enumerate(lines[begin + 1 :], start=begin + 1) if line.strip() == "__VSS_PORT_PROBE_END__")
    rows = []
    for line in lines[begin + 1 : end]:
        parts = ansi_escape.sub("", line).strip().split("|", 3)
        if len(parts) == 4 and parts[0].isdigit():
            rows.append(parts)

    print(f"{'PORT':>5}  {'STATUS':<13}  {'HTTP':>4}  NOTE")
    print(f"{'----':>5}  {'------':<13}  {'----':>4}  ----")
    for port, status, http_code, note in rows:
        print(f"{port:>5}  {status:<13}  {http_code:>4}  {note}")
except StopIteration:
    print(result.stdout)

if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f"nemohermes {NEMOCLAW_SANDBOX_NAME} connect probe failed with exit code {result.returncode}"
    )


## Standalone Hermes note

For Hermes directly on the VSS host, register this repository's VSS skills instead of using this managed notebook.

Primary setup:

```bash
hermes config edit
```

Add this repository's skills directory. You can copy the block from `.hermes/config.example.yaml`, replacing the placeholder with this checkout's absolute path:

```yaml
skills:
  external_dirs:
    - /path/to/video-search-and-summarization/skills
```

OpenClaw migration is optional and only needed when the user wants to bring compatible existing OpenClaw user data into Hermes:

```bash
hermes claw migrate --dry-run
hermes claw migrate --preset user-data --skill-conflict rename
```

Full migration, including compatible provider secrets:

```bash
hermes claw migrate --preset full --migrate-secrets --yes
```

Standalone Hermes runs on the host and can use shell, Docker, compose, and VSS APIs directly. The VSS Orchestrator MCP server is optional in that path.
